# Gradient Accumulation

Calling `backward()` **multiple times** without `zero_grad()` **adds** gradients — useful for large effective batch sizes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Three backward passes without zeroing

In [ ]:
model = SimpleMLP(in_dim=4, hidden=8, n_classes=2)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
grad_norms_per_step = []

for step in range(3):
    x = torch.randn(4, 4)
    y = torch.randint(0, 2, (4,))
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    total_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
    grad_norms_per_step.append(total_norm)
    print(f"Step {step+1}: loss={loss.item():.3f}, total grad norm={total_norm:.3f}")

## 2. Bar chart — accumulated gradient norm grows

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 4), grad_norms_per_step, color='steelblue', edgecolor='black')
ax.set_xlabel('backward() call #'); ax.set_ylabel('total ||grad||')
ax.set_title('Gradients accumulate across backward() calls')
ax.set_xticks([1, 2, 3])
plt.tight_layout(); plt.show()

## 3. Compare: with vs without `zero_grad()`

In [ ]:
def run_accumulate(zero_between=False, n=3):
    m = SimpleMLP(in_dim=4, hidden=8, n_classes=2)
    norms = []
    for _ in range(n):
        if zero_between:
            m.zero_grad()
        loss = F.cross_entropy(m(torch.randn(4, 4)), torch.randint(0, 2, (4,)))
        loss.backward()
        norms.append(sum(p.grad.norm().item() for p in m.parameters() if p.grad is not None))
    return norms

acc = run_accumulate(zero_between=False)
reset = run_accumulate(zero_between=True)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, 4)
w = 0.35
ax.bar(x - w/2, acc, w, label='accumulate', color='coral')
ax.bar(x + w/2, reset, w, label='zero_grad each step', color='seagreen')
ax.set_xlabel('backward() step'); ax.set_ylabel('total ||grad||')
ax.legend(); ax.set_title('zero_grad() resets accumulation')
plt.tight_layout(); plt.show()

## 4. Effective batch size via accumulation
Simulate batch_size=32 with micro-batches of 8.

In [ ]:
model = SimpleMLP(in_dim=4, hidden=8, n_classes=2)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
micro = 8
steps = 4
model.zero_grad()
losses = []
for s in range(steps):
    loss = F.cross_entropy(model(torch.randn(micro, 4)), torch.randint(0, 2, (micro,)))
    (loss / steps).backward()
    losses.append(loss.item())
opt.step()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, steps+1), losses, 'o-', color='purple')
axes[0].set_title('Micro-batch losses'); axes[0].set_xlabel('micro-step')
axes[1].bar(['single optimizer.step'], [sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)],
            color='teal')
axes[1].set_title('One weight update after 4 accumulated grads')
plt.tight_layout(); plt.show()